# Softmax Function

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Normalization
from sklearn.datasets import make_blobs


In [ ]:
def soft_max(z):
    z = np.asarray(z)
    ez = np.exp(z - np.max(z, axis=-1, keepdims=True))
    sm = ez / np.sum(ez, axis=-1, keepdims=True)
    return sm

### Data set



In [ ]:
# make  dataset for example
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
X_train, y_train = make_blobs(n_samples=2000, centers=centers, cluster_std=1.0,random_state=30)
# print(X_train)
print(f"max: {X_train[:,0].max()} , {X_train[:,1].max()},  min: {X_train[:,0].min()} , {X_train[:,1].min()}")

### 3. Normalizing data

In [ ]:
norm_layer = Normalization(axis=-1)
norm_layer.adapt(X_train)
Xnorm = norm_layer(X_train).numpy()
# print(f"max: {Xnorm[:,0].max()} , {Xnorm[:,1].max()},  min: {Xnorm[:,0].min()} , {Xnorm[:,1].min()}")

### 4. Creating Non Preferred Model

In [ ]:
tf.random.set_seed(12393)
model = Sequential([
    tf.keras.Input(shape=(2,)),
    Dense(units = 25, activation='relu', name = 'layer-1'),
    Dense(units = 15, activation='relu', name = 'layer-2'),
    Dense(units = 4, activation = 'softmax', name = 'output-layer')
])

model.summary()

model.compile(
    loss = tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer = tf.keras.optimizers.Adam(0.01)
)

model.fit(Xnorm, y_train, epochs = 10, batch_size=32)

# Predicting value
p_nonpreferred = model.predict(Xnorm)
print(p_nonpreferred [:2])
print("largest value", np.max(p_nonpreferred), "smallest value", np.min(p_nonpreferred))

### 5. Creating Preferred Model

In [ ]:

preferred_model = Sequential([
    tf.keras.Input(shape = (2,)),
    Dense(units = 25, activation = 'relu', name ='layer-1'),
    Dense(units = 15, activation = 'relu', name = 'layer-2'),
    Dense(units  = 4, activation = 'linear', name = 'outputlayer')
])

preferred_model.compile(
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),
    optimizer = tf.keras.optimizers.Adam(0.01)
)

preferred_model.fit(Xnorm, y_train, epochs = 20, batch_size = 32)

p_preferred = preferred_model.predict(Xnorm)
print(f"two example output vectors:\n {p_preferred[:2]}")
print("largest value", np.max(p_preferred), "smallest value", np.min(p_preferred))

In [ ]:
sm_preferred = tf.nn.softmax(p_preferred).numpy()
print(f"two example output vectors:\n {sm_preferred[:2]}")
print("largest value", np.max(sm_preferred), "smallest value", np.min(sm_preferred))


In [ ]:
for i in range(5):
    print( f"{p_preferred[i]}, category: {np.argmax(p_preferred[i])}")
    